In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
spark = SparkSession.builder.appName("data-cleaning").getOrCreate()


In [2]:
my_df_data_set = spark.read.csv("ecommerce_uncleaned_data.csv", header= True, inferSchema=True)
my_df_data_set.printSchema()
my_df_data_set.columns



root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- stock_on_hand: string (nullable = true)
 |-- supplier: string (nullable = true)



['order_id',
 'order_date',
 'customer_name',
 'email',
 'product_name',
 'quantity',
 'unit_price',
 'payment_status',
 'stock_on_hand',
 'supplier']

In [4]:
my_df_data_set.schema.fields

[StructField('order_id', StringType(), True),
 StructField('order_date', StringType(), True),
 StructField('customer_name', StringType(), True),
 StructField('email', StringType(), True),
 StructField('product_name', StringType(), True),
 StructField('quantity', StringType(), True),
 StructField('unit_price', StringType(), True),
 StructField('payment_status', StringType(), True),
 StructField('stock_on_hand', StringType(), True),
 StructField('supplier', StringType(), True)]

In [5]:
my_df_data_set.show(3)

+---------+----------+-------------+--------------------+-----------------+--------+----------+--------------+-------------+---------------+
| order_id|order_date|customer_name|               email|     product_name|quantity|unit_price|payment_status|stock_on_hand|       supplier|
+---------+----------+-------------+--------------------+-----------------+--------+----------+--------------+-------------+---------------+
| ORD10001|01/02/2025|    Manoj Das|manoj.das@rediffm...|   USB-C Cable 1m|       1|   ₹299.00|          Paid|         NULL|Sunrise Exports|
| ORD10002|2025-01-02| Vikram Singh|vikramsingh@gmail...|   Wireless Mouse|       2|    799.00|          Paid|          178|   Acme Traders|
|ORD-10003|02/01/2025|          N/A|naveenprasad@gmai...|Gel Pens (10 pk) |       1|      NULL|          NULL|          499|   Acme Traders|
+---------+----------+-------------+--------------------+-----------------+--------+----------+--------------+-------------+---------------+
only showing 

In [6]:
own_schema = t.StructType([
      t.StructField("order_id", t.StringType(), False),
      t.StructField("order_date", t.DateType(), True),
      t.StructField("customer_name", t.StringType(), True),
      t.StructField("email", t.StringType(), True),
      t.StructField("product_name", t.StringType(), True),
      t.StructField("quantity", t.IntegerType(), True),
      t.StructField("unit_price", t.DoubleType(), True),
      t.StructField("payment_status", t.StringType(), True),
       t.StructField("stock_on_hand", t.StringType(), True),
       t.StructField("supplier", t.StringType(), True),

]
)
custom_schema_df = spark.read.csv("ecommerce_uncleaned_data.csv", schema=own_schema)
custom_schema_df.filter(f.col("order_id").isNotNull()).show(2)
total_rows = custom_schema_df.count()
print(f"Total rows presented are: {total_rows}")

+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
|order_id|order_date|customer_name|               email|  product_name|quantity|unit_price|payment_status|stock_on_hand|       supplier|
+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
|order_id|      NULL|customer_name|               email|  product_name|    NULL|      NULL|payment_status|stock_on_hand|       supplier|
|ORD10001|      NULL|    Manoj Das|manoj.das@rediffm...|USB-C Cable 1m|       1|      NULL|          Paid|         NULL|Sunrise Exports|
+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
only showing top 2 rows
Total rows presented are: 277


In [ ]:
target_columns = custom_schema_df.columns
df_drop_all_col_empty = custom_schema_df.dropna(how = "all")
df_drop_all_col_empty.show(2)
df_drop_all_col_empty.write.mode("overwrite").options(header = True).csv("df_drop_all_col_empty.csv")


+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
|order_id|order_date|customer_name|               email|  product_name|quantity|unit_price|payment_status|stock_on_hand|       supplier|
+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
|order_id|      NULL|customer_name|               email|  product_name|    NULL|      NULL|payment_status|stock_on_hand|       supplier|
|ORD10001|      NULL|    Manoj Das|manoj.das@rediffm...|USB-C Cable 1m|       1|      NULL|          Paid|         NULL|Sunrise Exports|
+--------+----------+-------------+--------------------+--------------+--------+----------+--------------+-------------+---------------+
only showing top 2 rows


In [7]:
df_data_set_drop_all_col = my_df_data_set.dropna(how="all")


In [46]:


data_clean = df_data_set_drop_all_col.withColumn("order_id",
    f.concat(
      f.lit('ORD'),
      f.lpad(
        f.regexp_extract(
          f.col("order_id"), r"(\d+)", 1),
        5, "0"
      )
    )
).withColumn(
        "order_date", f.when(f.col("order_date").isNotNull(),
        f.coalesce(
          f.try_to_timestamp("order_date", f.lit("yyyy-MM-dd")),
        f.try_to_timestamp("order_date", f.lit("dd-MMM-yyyy")),
        f.try_to_timestamp("order_date", f.lit("MM/dd/yyyy")),
        f.try_to_timestamp("order_date", f.lit("dd/MM/yyyy"))
    ).cast("date")).when(f.col("order_date").isNull(), f.lit("1990-01-01").cast("date"))
).withColumn(
    "customer_name",
    f.trim(
        f.initcap(

              f.col("customer_name")
        )
    )
).withColumn(
    "email",
    f.trim(f.lower(
        f.regexp_replace(
          f.col("email"),
             "^mailto:", ""
        )
    ))
).withColumn(
    "product_name",
    f.trim(f.initcap(
        f.regexp_replace(
          f.col("product_name"), "[^a-z A-Z]", ""
        )

    ))
).withColumn("quantity",
             f.trim(f.regexp_extract(f.col("quantity"), r"(\d+)", 0))

).withColumn("unit_price",
             f.trim(f.regexp_extract(f.col("unit_price"), r"(\d+\.?\d*)", 0))

).withColumn("payment_status",
            f.trim(
                f.initcap(
                    f.regexp_replace(f.col("payment_status"), "COD - Pending", "Pending")
                )
            )
).withColumn("stock_on_hand",
             f.trim(
                 f.regexp_extract(f.col("stock_on_hand"), r"(\d+)", 0)
             )
).withColumn("supplier",
             f.trim(f.initcap(
                 f.regexp_replace(
                   f.col("supplier"), "[^a-z A-Z]", ""
                 )
             ))
             )
data_clean.show(2, truncate=False)
data_clean.write.mode("overwrite").options(headers = True).csv("df_drop_all_col_empty.csv")

+--------+----------+-------------+------------------------+--------------+--------+----------+--------------+-------------+---------------+
|order_id|order_date|customer_name|email                   |product_name  |quantity|unit_price|payment_status|stock_on_hand|supplier       |
+--------+----------+-------------+------------------------+--------------+--------+----------+--------------+-------------+---------------+
|ORD10001|2025-01-02|Manoj Das    |manoj.das@rediffmail.con|Usbc Cable M  |1       |299.00    |Paid          |NULL         |Sunrise Exports|
|ORD10002|2025-01-02|Vikram Singh |vikramsingh@gmail.com   |Wireless Mouse|2       |799.00    |Paid          |178          |Acme Traders   |
+--------+----------+-------------+------------------------+--------------+--------+----------+--------------+-------------+---------------+
only showing top 2 rows


In [63]:
df_drop_all_col_empty_default_fill = data_clean.fillna({
                                   "order_id" : 0,
                                   "customer_name": "Unknown",
                                   "email": "Not mentioned",
                                   "product_name": "product is empty",
                                   "quantity": 0,
                                   "unit_price": 0.0,
                                   "payment_status": "Unknown",
                                   "stock_on_hand": 0,
                                   "supplier": "Unknown"
                                   })
df_drop_all_col_empty_default_fill_with_date = df_drop_all_col_empty_default_fill.withColumn("order_date",
                                                           f.when(
                                                            f.col("order_date").isNull(), f.lit("1990-01-01").cast("date")
                                                          ).otherwise(f.col("order_date"))

                                                           ).dropDuplicates()

df_drop_all_col_empty_default_fill_with_date.show(50)
df_drop_all_col_empty_default_fill_with_date.write.mode("overwrite").options(header = True).csv("df_drop_all_col_empty.csv")

+--------+----------+------------------+--------------------+------------------+--------+----------+--------------+-------------+--------------------+
|order_id|order_date|     customer_name|               email|      product_name|quantity|unit_price|payment_status|stock_on_hand|            supplier|
+--------+----------+------------------+--------------------+------------------+--------+----------+--------------+-------------+--------------------+
|ORD10066|2025-03-02|      Deepak Patel|deepakpatel@yahoo...|      Face Wash Ml|       5|    249.00|   In Progress|          339|             Unknown|
|ORD10098|2025-03-20|        Divya Nair|divya.nair@gmail.com|          Yoga Mat|       0|    899.00|       Pending|          106|    Zenith Wholesale|
|ORD10144|2025-04-22|       Suresh Babu|sureshbabu@yahoo....|      Usbc Cable M|       1|       0.0|          Paid|          398|     Sunrise Exports|
|ORD10017|2025-01-14|     Sharma, Priya|priyasharma80@hot...|      Nonstick Pan|       3|   11

In [ ]:
df_drop_all_col_empty_default_fill.printSchema()

root
 |-- order_id: integer (nullable = false)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- payment_status: string (nullable = false)
 |-- stock_on_hand: string (nullable = false)
 |-- supplier: string (nullable = false)



In [ ]:
df_remove_duplicate = df_drop_all_col_empty_default_fill.distinct().drop()
df_remove_duplicate.show(35)
df_remove_duplicate.write.mode("overwrite").options(header = True).csv("df_remove_duplicate")


+--------+----------+--------------------+--------------------+------------------+--------+----------+--------------+-------------+------------------+
|order_id|order_date|       customer_name|               email|      product_name|quantity|unit_price|payment_status|stock_on_hand|          supplier|
+--------+----------+--------------------+--------------------+------------------+--------+----------+--------------+-------------+------------------+
|       0|2025-04-05|                 N/A|sneha_iyer27@gmai...|    Cotton T-Shirt|       1|     499.0|          Paid|          278|Zenith Wholesalers|
|       0|2025-05-25|         pooja reddy|pooja_reddy12@hot...|     Running Shoes|       1|       0.0|       Success|           73|   Delta Logistics|
|       0|2025-06-08|      Fatima  Begum |fatima_begum@outl...|          Yoga Mat|       1|     899.0|          Paid|           80|Zenith Wholesalers|
|       0|2025-03-16|   Lakshmi Narayanan|lakshminarayanan@...|     Wireles Mouse|       0|   

SUMMARY & TECHNIQUES USED:
Data cleaning preformed for the uncleaned data for E-commerce. Below are the techniques used:
StructType, StructField
isNotNull() and isNull() for null handing
trim, initcap, upper, lower, regexp_extract and regexp_replace to perform invaild string

